# ASC26 赛题二：UnifoLM-WMA-0 复现与推理优化（Colab）

**复现场景**：`unitree_z1_dual_arm_cleanup_pencils`、`unitree_z1_dual_arm_stackbox`（各 4 个 case）

**运行前**：Runtime → Change runtime type → GPU（优先 A100/L4）。

**说明**：所有结果必须来自真实运行，notebook 不预置任何数值。运行完成后用 Share → Anyone with the link 获取链接，填入 proposal。本 notebook 默认仓库为 `https://github.com/Radiant927/ASC`，仓库名不同请改第 2 节。

## 1. 环境检查

In [ ]:
!nvidia-smi
!df -h /content | tail -n 1

## 2. 克隆代码并安装依赖

In [ ]:
%cd /content
!git clone https://github.com/Radiant927/ASC.git /content/ASC 2>/dev/null || echo "ASC repo already cloned"
!git clone https://github.com/unitreerobotics/unifolm-world-model-action.git /content/unifolm-world-model-action 2>/dev/null || echo "model repo already cloned"
%cd /content/unifolm-world-model-action
!pip install -q -r requirements.txt

## 3. 下载权重与输入数据

权重约 22 GB，确认 Colab 磁盘空间充足。输入仓库的目录结构以官方 README 为准，若场景目录不在预期位置，按 README 把场景与参考视频放到 `unifolm-world-model-action` 仓库目录。

In [ ]:
!pip install -q -U huggingface_hub
!mkdir -p /content/checkpoints
!huggingface-cli download unitreerobotics/UnifoLM-WMA-0-Dual --local-dir /content/checkpoints/UnifoLM-WMA-0-Dual
!git clone https://github.com/ASC-Competition/ASC26-Embodied-World-Model-Optimizations.git /content/asc26-inputs 2>/dev/null || echo "inputs already cloned"
!find /content/asc26-inputs -maxdepth 2 | head -50

## 4. 配置 data_dir

按官方 README 调整数据目录后，把配置中的 `data_dir` 改为输入数据的绝对路径。

In [ ]:
from pathlib import Path
import yaml

REPO = Path("/content/unifolm-world-model-action")
DATA_DIR = "/content/asc26-inputs"
cfg_path = REPO / "configs/inference/world_model_interaction.yaml"
cfg = yaml.safe_load(cfg_path.read_text())

def set_data_dir(node):
    hits = 0
    if isinstance(node, dict):
        for key, value in node.items():
            if key == "data_dir":
                node[key] = DATA_DIR
                hits += 1
            else:
                hits += set_data_dir(value)
    elif isinstance(node, list):
        for value in node:
            hits += set_data_dir(value)
    return hits

print("updated data_dir fields:", set_data_dir(cfg))
cfg_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

## 5. 生成优化配置并运行 baseline + optimized

每个样本需完整跑基线，耗时较长。Colab 空闲会断开，运行期间保持页面活跃。

In [ ]:
CODE = "/content/ASC/task2/Radiant_worldmodel/code"
RESULTS = "/content/ASC/task2/Radiant_worldmodel/results"
!python {CODE}/prepare_opt_config.py --repo-dir /content/unifolm-world-model-action --steps 25

In [ ]:
!REPO_DIR=/content/unifolm-world-model-action RESULTS_DIR={RESULTS} SCENARIOS="unitree_z1_dual_arm_cleanup_pencils,unitree_z1_dual_arm_stackbox" CASES="case1,case2,case3,case4" bash {CODE}/run_all.sh

## 6. PSNR 评估

先查看参考视频的实际路径，再按需修改下面的 glob 规则。

In [ ]:
from pathlib import Path
for path in sorted(Path("/content/asc26-inputs").rglob("*.mp4"))[:60]:
    print(path)

In [ ]:
import subprocess
from pathlib import Path

SCENARIOS = ["unitree_z1_dual_arm_cleanup_pencils", "unitree_z1_dual_arm_stackbox"]
CASES = ["case1", "case2", "case3", "case4"]
GT_ROOT = Path("/content/asc26-inputs")
RESULTS = Path("/content/ASC/task2/Radiant_worldmodel/results")

for scenario in SCENARIOS:
    for case in CASES:
        pred = RESULTS / scenario / case / "0_full_fs6.mp4"
        candidates = [p for p in GT_ROOT.rglob("*.mp4") if scenario in str(p) and case in str(p)]
        if not pred.exists() or not candidates:
            print(f"skip {scenario}/{case}: pred={pred.exists()} gt_candidates={len(candidates)}")
            continue
        gt = sorted(candidates)[0]
        subprocess.run([
            "python", f"{CODE}/psnr_eval.py",
            "--gt", str(gt), "--pred", str(pred),
            "--output", str(RESULTS / scenario / case / "psnr.json"),
        ], check=True)

## 7. 汇总、PSNR 图表与 proposal

In [ ]:
!cd {CODE} && python gen_summary.py --results {RESULTS} --scenarios unitree_z1_dual_arm_cleanup_pencils,unitree_z1_dual_arm_stackbox --cases case1,case2,case3,case4 --output {RESULTS}/summary.json
!cd {CODE} && python plot_psnr.py --summary {RESULTS}/summary.json --output {RESULTS}/psnr_chart.png
!cd {CODE} && python make_proposal.py --summary {RESULTS}/summary.json --chart {RESULTS}/psnr_chart.png --output /content/ASC/task2/Radiant_worldmodel/proposal_file/ASC26_Task2_Proposal.docx --team Radiant

## 8. 打包结果

下载压缩包并提交到仓库；同时把本 notebook 设为 Anyone with the link 可查看，链接填入 proposal 与 README。

In [ ]:
!cd /content/ASC && zip -qr /content/Radiant_worldmodel.zip task2/Radiant_worldmodel
from google.colab import files
files.download("/content/Radiant_worldmodel.zip")